# Raw data preprocessing

Converts raw pump output into the three arrays used by the analysis notebook:
`time` (minutes), `volume_1` (biotic), `volume_2` (sterile), saved as `.npy`.

Two input formats are handled:

- **Qizix pump (`.dat`)** &mdash; Section 2. Trim the header/footer, then extract the columns.
- **Vindum pump (`.csv`)** &mdash; Section 3. Optionally split (if cylinders were switched
  mid-experiment), trim, parse, and stitch the two halves back together.

Both pipelines end by saving `*-time.npy`, `*-volume_1.npy`, `*-volume_2.npy` into the
arrays folder set in the configuration cell below. Edit **only** the configuration cell
between experiments.


## 0. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


## 1. Configuration

All paths and per-experiment settings live here. Nothing below this cell should need editing
between runs (the `.dat` pipeline is driven by widgets that default to these values; the
`.csv` pipeline reads them directly).

In [ ]:
# ---- folders ----
DATA_DIR    = ""
TRIM_DIR    = f"{DATA_DIR}/trimmed output"      # intermediate trimmed .dat files
ARRAYS_DIR  = f"{DATA_DIR}/the column"          # final .npy arrays land here

# ---- current experiment ----
PREFIX      = "2026-02-25_h2-400bar_SRB"        # base name for the output arrays

# ---- Qizix .dat settings ----
DAT_INPUT     = f"{DATA_DIR}/2026-01-29_h2-400bar_SRB.csv"
DAT_SKIP_TOP  = 6
DAT_SKIP_BOTTOM = 8
DAT_USECOLS   = [2, 11, 12]    # (time, volume_1, volume_2) column indices in the .dat

# ---- Vindum .csv settings ----
CSV_INPUT     = f"{DATA_DIR}/2026-02-25_h2-400bar_SRB.csv"
CSV_COLS      = ["Date", "Time", "P1 Vol A", "P1 Vol B", "P2 Vol A", "P2 Vol B"]
CSV_DATETIME_FMT = "%d-%m-%Y %H:%M:%S"

# scratch files written by the .csv split/trim steps (kept in DATA_DIR)
BEFORE_CSV  = f"{DATA_DIR}/bef.csv"
AFTER_CSV   = f"{DATA_DIR}/af.csv"

os.makedirs(TRIM_DIR, exist_ok=True)
os.makedirs(ARRAYS_DIR, exist_ok=True)
print("Config loaded. Prefix:", PREFIX)


## 2. Shared helpers

Defined once and reused by both pipelines: a plot, an array-saver, and the array stitcher.

In [ ]:
def plot_consumption(time, v1, v2, title="Hydrogen consumption of SRB",
                     xlabel="Time [minutes]", label1="Biotic", label2="Sterile control"):
    """Standard biotic-vs-sterile consumption plot."""
    plt.figure(figsize=(8, 6))
    plt.plot(time, v1, label=label1, linewidth=1)
    plt.plot(time, v2, label=label2, linewidth=1)
    plt.xlabel(xlabel)
    plt.ylabel("H2 consumed (ml)")
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.show()


def save_arrays(prefix, time, v1, v2, output_folder=None):
    """Save the three arrays as <prefix>-time/-volume_1/-volume_2 .npy."""
    output_folder = output_folder or ARRAYS_DIR
    os.makedirs(output_folder, exist_ok=True)
    names = {
        "time":     f"{prefix}-time.npy",
        "volume_1": f"{prefix}-volume_1.npy",
        "volume_2": f"{prefix}-volume_2.npy",
    }
    np.save(os.path.join(output_folder, names["time"]),     np.asarray(time))
    np.save(os.path.join(output_folder, names["volume_1"]), np.asarray(v1))
    np.save(os.path.join(output_folder, names["volume_2"]), np.asarray(v2))
    print(f"Saved -> {output_folder}")
    for n in names.values():
        print(f"   {n}")


def auto_stitch(array1, array2):
    """Join two cumulative series, shifting the second so it continues from the first."""
    a1 = np.asarray(array1, float)
    a2 = np.asarray(array2, float)
    offset = a1[-1] - a2[0]
    return np.concatenate([a1, a2 + offset])


## 3. Qizix pump (`.dat` files)

**3a. Trim** removes the header and footer rows from the raw `.dat`.
**3b. Extract** reads the time/volume columns and saves the `.npy` arrays.

Both steps are driven by the widget below, pre-filled from the configuration cell.

In [ ]:
def trim_dat_file(input_file, output_folder, skip_top=6, skip_bottom=8):
    """Remove the first `skip_top` and last `skip_bottom` lines of a .dat file.
    Writes <name>-trimmed<ext> into output_folder and returns its path."""
    _, filename = os.path.split(input_file)
    name, ext = os.path.splitext(filename)
    output_file = os.path.join(output_folder, f"{name}-trimmed{ext}")
    os.makedirs(output_folder, exist_ok=True)
    try:
        with open(input_file, "r") as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: input file not found at {input_file}")
        return None
    if len(lines) <= skip_top + skip_bottom:
        print(f"Error: file has only {len(lines)} lines, fewer than the skip count.")
        return None
    with open(output_file, "w") as f:
        f.writelines(lines[skip_top: len(lines) - skip_bottom])
    print(f"Trimmed file saved: {output_file}")
    return output_file


def extract_time_and_volume(input_path, output_path, output_folder, usecols=DAT_USECOLS):
    """Read the (time, volume_1, volume_2) columns from a trimmed .dat, drop bad rows,
    save a reduced .dat and the three .npy arrays. Returns the arrays as a dict."""
    file_prefix = os.path.splitext(os.path.basename(input_path))[0].replace("-trimmed", "")
    print(f"Prefix for naming: {file_prefix}")
    os.makedirs(output_folder, exist_ok=True)
    try:
        df = pd.read_csv(input_path, header=None, usecols=usecols,
                         delimiter=",", engine="python")
    except Exception as e:
        print(f"Error reading file: {e}")
        return None
    df.columns = ["time", "volume_1", "volume_2"]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=df.columns)

    dat_folder = os.path.dirname(output_path)
    if dat_folder:
        os.makedirs(dat_folder, exist_ok=True)
    df.to_csv(output_path, index=False, header=False, float_format="%.6f")

    save_arrays(file_prefix, df["time"], df["volume_1"], df["volume_2"], output_folder)
    return {c: df[c].to_numpy() for c in df.columns}


In [ ]:
# ---- widget: trim then extract a .dat file ----
style = {"description_width": "120px"}
layout_full = widgets.Layout(width="90%")

w_input_path   = widgets.Text(value=DAT_INPUT,  description="<b>Raw .dat:</b>",
                              style=style, layout=layout_full)
w_output_folder = widgets.Text(value=TRIM_DIR,  description="<b>Trimmed folder:</b>",
                              style=style, layout=layout_full)
w_arrays_folder = widgets.Text(value=ARRAYS_DIR, description="<b>Arrays folder:</b>",
                              style=style, layout=layout_full)
w_skip_top    = widgets.IntText(value=DAT_SKIP_TOP,    description="Skip top:",    style=style)
w_skip_bottom = widgets.IntText(value=DAT_SKIP_BOTTOM, description="Skip bottom:", style=style)

w_btn_run = widgets.Button(description="Trim + Extract", button_style="success", icon="cut",
                           layout=widgets.Layout(width="100%", margin="15px 0 0 0"))
w_out = widgets.Output()

def on_run_click(b):
    w_out.clear_output()
    with w_out:
        print("Processing...")
        trimmed = trim_dat_file(w_input_path.value, w_output_folder.value,
                                w_skip_top.value, w_skip_bottom.value)
        if trimmed is None:
            return
        base = os.path.splitext(os.path.basename(trimmed))[0].replace("-trimmed", "")
        reduced = os.path.join(w_output_folder.value, f"{base}-final_reduced.dat")
        extract_time_and_volume(trimmed, reduced, w_arrays_folder.value)
        print("Done.")

w_btn_run.on_click(on_run_click)

display(widgets.VBox([
    widgets.HTML("<h3>Qizix .dat: trim + extract</h3>"),
    w_input_path, w_output_folder, w_arrays_folder,
    widgets.HBox([w_skip_top, w_skip_bottom]),
    w_btn_run, w_out,
]))


**Quick check** — load the arrays you just saved and plot them.

In [ ]:
prefix = PREFIX   # or set to the file you just processed
time     = np.load(os.path.join(ARRAYS_DIR, f"{prefix}-time.npy"))
volume_1 = np.load(os.path.join(ARRAYS_DIR, f"{prefix}-volume_1.npy"))
volume_2 = np.load(os.path.join(ARRAYS_DIR, f"{prefix}-volume_2.npy"))
plot_consumption(time, volume_1, volume_2)


## 4. Vindum pump (`.csv` files)

Use this when the experiment was logged by the Vindum pumps. The full chain is:

1. **(optional) split** the file in two, if the cylinders were switched mid-experiment
   to check for a leak &mdash; by timestamp *or* by row index.
2. **(optional) trim** unwanted rows from the start/end of a half.
3. **parse** each half into `Minutes` + volume arrays.
4. **stitch** the two halves back into one continuous series and save the `.npy` arrays.

If there was no switch, skip steps 1 and 4 and just parse the single file.

### 4a. (optional) Split a switched-cylinder file

In [ ]:
def split_csv_by_datetime(file_path, split_date, split_time,
                          before_path=BEFORE_CSV, after_path=AFTER_CSV,
                          fmt=CSV_DATETIME_FMT):
    """Split a raw Vindum .csv at a given date+time into before/after files."""
    df = pd.read_csv(file_path, sep=";", decimal=",", index_col=False)
    dt = pd.to_datetime(df["Date"] + " " + df["Time"], format=fmt)
    split_point = pd.to_datetime(f"{split_date} {split_time}", format=fmt)
    df[dt < split_point].to_csv(before_path, index=False)
    df[dt >= split_point].to_csv(after_path, index=False)
    print(f"Split by datetime -> before: {(dt < split_point).sum()}, "
          f"after: {(dt >= split_point).sum()} rows")


def split_csv_by_index(file_path, split_index,
                       before_path=BEFORE_CSV, after_path=AFTER_CSV):
    """Split a raw Vindum .csv at a given row index into before/after files."""
    df = pd.read_csv(file_path, sep=";", decimal=",", index_col=False)
    df.iloc[:split_index].to_csv(before_path, index=False)
    df.iloc[split_index:].to_csv(after_path, index=False)
    print(f"Split at index {split_index} -> before: {split_index}, "
          f"after: {len(df) - split_index} rows")


# --- run one of these as needed (edit the arguments) ---
# split_csv_by_datetime(CSV_INPUT, "6-3-2026", "11:57:51")
# split_csv_by_index(CSV_INPUT, 9980)


### 4b. (optional) Trim rows from a half

In [ ]:
def trim_csv(input_path, output_path, trim_start=0, trim_end=0, use_percentage=False):
    """Remove rows from the top/bottom of a CSV. trim_* are fractions if use_percentage,
    else row counts."""
    df = pd.read_csv(input_path)
    n = len(df)
    if use_percentage:
        start_idx, end_idx = int(n * trim_start), n - int(n * trim_end)
    else:
        start_idx, end_idx = int(trim_start), n - int(trim_end)
    df_trimmed = df.iloc[start_idx:end_idx].reset_index(drop=True)
    df_trimmed.to_csv(output_path, index=False)
    print(f"Trimmed {n - len(df_trimmed)} of {n} rows -> {output_path}")


# example: drop the first 10 rows of the 'before' half
# trim_csv(BEFORE_CSV, BEFORE_CSV, trim_start=10, trim_end=0)


### 4c. Parse a half into time + volume arrays

`load_vindum_csv` replaces the two near-identical parsing cells from the old notebook.
Pass which cylinder column pair you want (`"A"` or `"B"`).

In [ ]:
def load_vindum_csv(path, cols=CSV_COLS, fmt=CSV_DATETIME_FMT, cylinder="A"):
    """Read a (split/trimmed) Vindum CSV and return (minutes, v1, v2) as numpy arrays.
    `cylinder` selects the 'P1 Vol A/P2 Vol A' or 'P1 Vol B/P2 Vol B' pair."""
    df = pd.read_csv(path, sep=",", decimal=".", usecols=cols, index_col=False)
    dt = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True, format=fmt)
    minutes = (dt - dt.iloc[0]).dt.total_seconds().to_numpy() / 60.0
    v1 = df[f"P1 Vol {cylinder}"].to_numpy()
    v2 = df[f"P2 Vol {cylinder}"].to_numpy()
    return minutes, v1, v2


# --- parse the two halves (cylinder A before the switch, B after) ---
time_A, v1_A, v2_A = load_vindum_csv(BEFORE_CSV, cylinder="A")
time_B, v1_B, v2_B = load_vindum_csv(AFTER_CSV,  cylinder="B")

plot_consumption(time_A / 1440.0, v1_A, v2_A, xlabel="Time [days]",
                 title="Before switch (cylinder A)")
plot_consumption(time_B / 1440.0, v1_B, v2_B, xlabel="Time [days]",
                 title="After switch (cylinder B)")


### 4d. Stitch the halves and save

In [ ]:
# join the two halves into one continuous cumulative series
result_1 = auto_stitch(v1_A, v1_B)
result_2 = auto_stitch(v2_A, v2_B)

# the two halves use different clocks, so use a simple running index as the time axis
prov_time = np.arange(len(result_1))

plot_consumption(prov_time, result_1, result_2, title="Stitched series")

save_arrays(PREFIX, prov_time, result_1, result_2)


### 4e. (optional) Also save as one combined CSV

Some downstream tools prefer a single CSV with named columns.

In [ ]:
combined_csv = os.path.join(DATA_DIR, f"{PREFIX}-combined.csv")
combined_data = np.column_stack((prov_time, result_1, result_2))
np.savetxt(combined_csv, combined_data, delimiter=",",
           header="time,volume_1,volume_2", comments="")
print("Saved:", combined_csv)
